# Custom surface site analysis

Edit **Cell 2 only**, then **Run All**.


In [15]:
# Imports (DO NOT EDIT)
import sys, os
sys.path.append(".")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../..')))

from ase.io import read, write, Trajectory
from acat.settings import CustomSurface
from acat.adsorption_sites import SlabAdsorptionSites
from ase.build import surface
from ase.visualize import view

import site_analysis as sa

In [16]:
# USER INPUT
name = 'pd332.xyz'
n_layers = 10
atoms = surface('Pd',(3,3,2),n_layers)
atoms.center(vacuum=10, axis=2)
adsorbate_height = 2.
site_bond_cutoff = 3.

In [17]:
#visualize slab
write(name, atoms)
view(atoms, viewer='x3d')

In [18]:

# Load slab
slab = read(name)
surface = CustomSurface(slab, n_layers=n_layers)
nslab = len(slab)


In [19]:

slabrat = slab.copy()
slabrat.rattle(stdev=0.3)

# Generate symmetry-unique site geometries
cas = SlabAdsorptionSites(slab, "fcc332", composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slab, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!
#cas = SlabAdsorptionSites(slabrat, surface, composition_effect=True)  # ACAT has surface, custom does not find them all!

single_geoms, single_sites_lists = sa.generate_unique_sites(
    slab,
    cas.get_sites(),
    nslab,
    site_bond_cutoff,
    adsorbate_height
)

print(f'There are {len(single_sites_lists)} unique sites out of {len(cas.get_sites())}.')
#print(cas.get_sites())

traj = Trajectory("unique_sites.traj", "w")
for g in single_geoms:
    traj.write(g)
traj.close()


There are 72 unique sites out of 72.


In [43]:
[cas.get_sites()[i]['morphology'] for i in range(len(cas.get_sites()))]
print(cas.get_sites())
with open('site.txt',"a") as f:
    f.write(str(cas.get_sites()))
    f.write("\n")

[{'site': 'ontop', 'surface': 'fcc332', 'morphology': 'corner', 'position': array([-7.13941764,  2.47345955, 15.80545551]), 'normal': array([-0.32444284,  0.27668579,  0.90453403]), 'indices': (28,), 'composition': 'Pd', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([28])}, {'site': 'bridge', 'surface': 'fcc332', 'morphology': 'corner', 'position': array([-8.47805845,  2.7112922 , 16.01279321]), 'normal': array([-0.07647191,  0.06521547,  0.99493668]), 'indices': (28, 29), 'composition': 'PdPd', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([28, 29])}, {'site': 'fcc', 'surface': 'fcc332', 'morphology': 'tc-cc-h', 'position': array([-8.32932058,  3.48821219, 16.08190577]), 'normal': array([ 0.13245323, -0.1129565 ,  0.98473193]), 'indices': (28, 29, 30), 'composition': 'PdPdPd', 'subsurf_index': None, 'subsurf_element': None, 'label': None, 'topology': array([28, 29, 30])}, {'site': 'hcp', 'surface': 'fcc332', 'morpho

In [21]:

# Extract 3-fold site graphs
admols, threefold_geom_indices = sa.classify_threefold_sites(
    single_geoms, single_sites_lists
)


In [22]:

# Graph isomorphism clustering
iso_mat, clusters = sa.cluster_isomorphic_graphs(admols)


In [23]:

# Update 3-fold site labels
type_map = sa.update_threefold_site_labels(
    single_sites_lists,
    clusters,
    threefold_geom_indices
)


In [24]:

# Write 3-fold-only trajectory
traj3 = Trajectory("threefold_sites.traj", "w")
for i in threefold_geom_indices:
    traj3.write(single_geoms[i])
traj3.close()


In [25]:

# Pairwise strict isomorphism (PRINT)
print("Pairwise strict isomorphism:")
for i in range(len(admols)):
    for j in range(i + 1, len(admols)):
        print(f"{i} vs {j} =", iso_mat[i, j])


Pairwise strict isomorphism:
0 vs 1 = False
0 vs 2 = False
0 vs 3 = False
0 vs 4 = False
0 vs 5 = False
0 vs 6 = False
0 vs 7 = False
0 vs 8 = False
0 vs 9 = False
0 vs 10 = False
0 vs 11 = False
0 vs 12 = False
0 vs 13 = False
0 vs 14 = False
0 vs 15 = False
0 vs 16 = False
0 vs 17 = False
0 vs 18 = False
0 vs 19 = False
0 vs 20 = False
0 vs 21 = False
0 vs 22 = False
0 vs 23 = False
0 vs 24 = False
0 vs 25 = False
0 vs 26 = False
0 vs 27 = False
0 vs 28 = False
0 vs 29 = False
0 vs 30 = False
0 vs 31 = False
0 vs 32 = False
0 vs 33 = False
0 vs 34 = False
0 vs 35 = False
0 vs 36 = False
0 vs 37 = False
0 vs 38 = False
0 vs 39 = False
0 vs 40 = False
0 vs 41 = False
0 vs 42 = False
0 vs 43 = False
0 vs 44 = False
0 vs 45 = False
0 vs 46 = False
0 vs 47 = False
0 vs 48 = False
0 vs 49 = False
0 vs 50 = False
0 vs 51 = False
0 vs 52 = False
0 vs 53 = False
0 vs 54 = False
0 vs 55 = False
0 vs 56 = False
0 vs 57 = False
0 vs 58 = False
0 vs 59 = False
0 vs 60 = False
0 vs 61 = False
0 vs

In [26]:

# Distinct 3-fold site types (PRINT)
print("Number of distinct 3-fold site types:", len(clusters))
for members in clusters.values():
    print("3-fold site type:", members)


Number of distinct 3-fold site types: 36
3-fold site type: [0]
3-fold site type: [1, 13, 14, 16, 19, 20, 21]
3-fold site type: [2]
3-fold site type: [3, 11]
3-fold site type: [4, 23, 24]
3-fold site type: [5]
3-fold site type: [6]
3-fold site type: [7, 8, 10]
3-fold site type: [9, 12]
3-fold site type: [15]
3-fold site type: [17, 25]
3-fold site type: [18, 22]
3-fold site type: [26, 27]
3-fold site type: [28, 29]
3-fold site type: [30]
3-fold site type: [31]
3-fold site type: [32, 33]
3-fold site type: [34]
3-fold site type: [35, 37, 39, 46, 51, 55]
3-fold site type: [36]
3-fold site type: [38]
3-fold site type: [40, 48]
3-fold site type: [41, 42, 43]
3-fold site type: [44, 47]
3-fold site type: [45, 53, 54]
3-fold site type: [49]
3-fold site type: [50]
3-fold site type: [52]
3-fold site type: [56]
3-fold site type: [57, 64, 65, 68]
3-fold site type: [58]
3-fold site type: [59, 67]
3-fold site type: [60, 69, 70]
3-fold site type: [61, 62, 63]
3-fold site type: [66]
3-fold site type: [7

In [27]:

# Updated 3-fold site labels (PRINT)
print("Updated 3-fold site labels per geometry:")
for geom_idx, label in type_map.items():
    print(f"Geometry {geom_idx} -> {label}")


Updated 3-fold site labels per geometry:
Geometry 0 -> 3fold1
Geometry 1 -> 3fold2
Geometry 13 -> 3fold2
Geometry 14 -> 3fold2
Geometry 16 -> 3fold2
Geometry 19 -> 3fold2
Geometry 20 -> 3fold2
Geometry 21 -> 3fold2
Geometry 2 -> 3fold3
Geometry 3 -> 3fold4
Geometry 11 -> 3fold4
Geometry 4 -> 3fold5
Geometry 23 -> 3fold5
Geometry 24 -> 3fold5
Geometry 5 -> 3fold6
Geometry 6 -> 3fold7
Geometry 7 -> 3fold8
Geometry 8 -> 3fold8
Geometry 10 -> 3fold8
Geometry 9 -> 3fold9
Geometry 12 -> 3fold9
Geometry 15 -> 3fold10
Geometry 17 -> 3fold11
Geometry 25 -> 3fold11
Geometry 18 -> 3fold12
Geometry 22 -> 3fold12
Geometry 26 -> 3fold13
Geometry 27 -> 3fold13
Geometry 28 -> 3fold14
Geometry 29 -> 3fold14
Geometry 30 -> 3fold15
Geometry 31 -> 3fold16
Geometry 32 -> 3fold17
Geometry 33 -> 3fold17
Geometry 34 -> 3fold18
Geometry 35 -> 3fold19
Geometry 37 -> 3fold19
Geometry 39 -> 3fold19
Geometry 46 -> 3fold19
Geometry 51 -> 3fold19
Geometry 55 -> 3fold19
Geometry 36 -> 3fold20
Geometry 38 -> 3fold21
G

In [28]:
# Site yaml file generated
print("All sites for the custom surfaces are saved in site.yaml")
sa.write_sites_yaml(single_sites_lists, clusters)


All sites for the custom surfaces are saved in site.yaml
